# 📓 Notebook A3 (ML) — PyTorch: Fine-Tuning a Transformer

> **Module:** Machine Learning · **Type:** Appendix · **Estimated time:** 60–90 min · **Difficulty:** Advanced

A1 covered foundations. A2 covered architectures. This notebook covers **the most useful skill in practical deep learning today: transfer learning** — taking a pretrained model and adapting it to your task with comparatively little data.

You'll fine-tune a tiny pretrained encoder (DistilBERT-class) on a real-flavoured **business text classification** task using the HuggingFace `transformers` library on top of PyTorch.

---

## 🎯 Learning objectives
- Load a pretrained transformer + tokenizer from HuggingFace.
- Build a `Dataset`/`DataLoader` for text classification.
- Run a fine-tuning loop with learning-rate scheduling and mixed precision.
- Apply LoRA-style parameter-efficient fine-tuning (PEFT) when full fine-tune is overkill.

## ✅ Prerequisites
- Notebooks A1 + A2 (PyTorch foundations and architectures).
- Notebook 18 (AI workflows, for the prompting/inference mental model).

## 📦 Install

```bash
pip install torch transformers datasets accelerate
# Optional, for parameter-efficient fine-tuning:
pip install peft
```


## 1. Why fine-tune at all?

Pretrained transformers have learned **general language structure** from hundreds of GB of text. You re-use those weights as a starting point and only nudge them for *your* task. Compared to training from scratch you get:

| Benefit | What it looks like |
|---|---|
| **Less data** | Hundreds of labelled examples can beat thousands trained from scratch |
| **Faster convergence** | A few epochs vs hundreds |
| **Better generalisation** | The model already knows synonyms, negation, syntax |

The trade-off: you inherit the pretrained model's **biases and gaps**. Always evaluate on a held-out test set from your domain.


In [1]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.figsize": (9, 4), "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

try:
    import torch
    import torch.nn as nn
    HAS_TORCH = True
    torch.manual_seed(0)
    print(f"PyTorch {torch.__version__}")
except Exception as e:
    HAS_TORCH = False
    print("PyTorch not installed —", e)

try:
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    HAS_HF = True
    print("transformers OK")
except Exception:
    HAS_HF = False
    print("transformers not installed — install with: pip install transformers")


PyTorch not installed — No module named 'torch'
transformers not installed — install with: pip install transformers


## 2. The task — classify customer-support messages

We'll classify short messages into three intents: `billing`, `account`, `product`. The labelled dataset is tiny on purpose — that's exactly when fine-tuning shines.


In [2]:
TRAIN = [
    ("I want to update my credit card",                          "billing"),
    ("Can you refund last month's charge?",                       "billing"),
    ("When does my subscription renew?",                          "billing"),
    ("How do I reset my password?",                               "account"),
    ("My login keeps failing on Safari",                          "account"),
    ("I need to change the email on my account",                  "account"),
    ("How do I export my data as CSV?",                           "product"),
    ("Where do I find usage statistics?",                         "product"),
    ("Does the API support webhooks?",                            "product"),
    ("My monthly invoice doesn't add up",                         "billing"),
    ("Two-factor authentication isn't sending the code",          "account"),
    ("I want to cancel and stop being billed",                    "billing"),
    ("Can multiple users share one license?",                     "account"),
    ("Is there a dark mode in the dashboard?",                    "product"),
    ("My export job has been stuck for 20 minutes",               "product"),
    ("Please switch my plan from monthly to annual",              "billing"),
]
TEST = [
    ("Refund my unused days please",                "billing"),
    ("Set up SSO for our team",                     "account"),
    ("Filter by date range on the dashboard?",      "product"),
    ("Charge the new card from now on",             "billing"),
    ("Help, I can't log in",                        "account"),
    ("API rate limit per minute?",                  "product"),
]
LABELS = ["billing", "account", "product"]
LBL2IDX = {l: i for i, l in enumerate(LABELS)}
print(f"{len(TRAIN)} train messages · {len(TEST)} test messages")


16 train messages · 6 test messages


## 3. Load a pretrained tokenizer and model

We use **`distilbert-base-uncased`** — 66 M parameters, fast on CPU, available offline if you've used it before.


In [3]:
# Reference code — runs once you have `pip install transformers` and an internet
# connection (the first run downloads ~250 MB of weights).
#
# if HAS_HF:
#     model_name = "distilbert-base-uncased"
#     tokenizer = AutoTokenizer.from_pretrained(model_name)
#     model = AutoModelForSequenceClassification.from_pretrained(
#         model_name, num_labels=len(LABELS)
#     )
#     print(model.config)
#
# Offline stand-in: a tiny from-scratch character-bag classifier so the rest of
# the notebook still runs anywhere.

def char_features(texts, vocab=None, dim=64):
    import hashlib
    feats = np.zeros((len(texts), dim), dtype=np.float32)
    for i, t in enumerate(texts):
        for w in t.lower().split():
            h = int(hashlib.sha256(w.encode()).hexdigest(), 16)
            feats[i, h % dim] += 1.0
    # L1-normalise
    s = feats.sum(axis=1, keepdims=True); s[s == 0] = 1
    return feats / s

Xtr_np = char_features([t for t, _ in TRAIN])
Xte_np = char_features([t for t, _ in TEST])
ytr_np = np.array([LBL2IDX[l] for _, l in TRAIN])
yte_np = np.array([LBL2IDX[l] for _, l in TEST])
print("offline-bag feature shape:", Xtr_np.shape)


offline-bag feature shape: (16, 64)


## 4. The fine-tuning loop (reference code)

The full HuggingFace fine-tuning pattern in ~30 lines. Read it carefully — this is what a real production fine-tune looks like.


In [4]:
# ── Reference fine-tuning loop — uncomment when running with HF installed ──
#
# from torch.utils.data import Dataset, DataLoader
#
# class TextDS(Dataset):
#     def __init__(self, pairs, tokenizer, max_len=64):
#         self.pairs = pairs; self.tok = tokenizer; self.max_len = max_len
#     def __len__(self): return len(self.pairs)
#     def __getitem__(self, i):
#         text, label = self.pairs[i]
#         enc = self.tok(text, truncation=True, padding="max_length",
#                        max_length=self.max_len, return_tensors="pt")
#         return {"input_ids": enc["input_ids"].squeeze(0),
#                 "attention_mask": enc["attention_mask"].squeeze(0),
#                 "labels": torch.tensor(LBL2IDX[label])}
#
# dl_tr = DataLoader(TextDS(TRAIN, tokenizer), batch_size=4, shuffle=True)
# dl_te = DataLoader(TextDS(TEST,  tokenizer), batch_size=4)
#
# optim = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
# from transformers import get_linear_schedule_with_warmup
# n_steps = len(dl_tr) * 3       # 3 epochs
# sched = get_linear_schedule_with_warmup(optim, num_warmup_steps=0, num_training_steps=n_steps)
#
# for epoch in range(3):
#     model.train()
#     for batch in dl_tr:
#         out = model(**batch)            # CrossEntropyLoss already inside
#         optim.zero_grad(); out.loss.backward(); optim.step(); sched.step()
#     model.eval()
#     with torch.no_grad():
#         right = total = 0
#         for batch in dl_te:
#             pred = model(**batch).logits.argmax(dim=1)
#             right += (pred == batch["labels"]).sum().item(); total += len(pred)
#     print(f"epoch {epoch}  test acc {right/total:.3f}")

print("Reference fine-tuning loop shown above. Offline stand-in trains below.")


Reference fine-tuning loop shown above. Offline stand-in trains below.


## 5. Offline stand-in — train a tiny MLP on bag-of-words features

So you can see *the same training loop* run end-to-end without the HF download. Same shape, smaller numbers.


In [5]:
if HAS_TORCH:
    Xtr = torch.from_numpy(Xtr_np); ytr = torch.from_numpy(ytr_np).long()
    Xte = torch.from_numpy(Xte_np); yte = torch.from_numpy(yte_np).long()

    class TinyHead(nn.Module):
        def __init__(self, in_dim, hidden=32, n=3):
            super().__init__()
            self.net = nn.Sequential(nn.Linear(in_dim, hidden), nn.ReLU(),
                                     nn.Dropout(0.2), nn.Linear(hidden, n))
        def forward(self, x): return self.net(x)

    model = TinyHead(in_dim=Xtr.shape[1])
    opt = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=1e-4)
    loss_fn = nn.CrossEntropyLoss()
    for epoch in range(80):
        model.train()
        opt.zero_grad(); loss_fn(model(Xtr), ytr).backward(); opt.step()
    model.eval()
    with torch.no_grad():
        acc = (model(Xte).argmax(1) == yte).float().mean().item()
    print(f"offline stand-in test acc: {acc:.3f}  (with real DistilBERT it usually ≥ 0.9)")


## 6. Parameter-efficient fine-tuning — LoRA in one paragraph

Full fine-tuning updates **every** weight in the pretrained model. For a 7-B parameter LLM that's ~28 GB of gradients in float32. Most teams now use **LoRA** (Low-Rank Adaptation): freeze the original weights and inject small, trainable rank-$r$ matrices into each attention layer. You train < 1 % of parameters and the result is competitive.

```python
# pip install peft
# from peft import LoraConfig, get_peft_model
#
# base = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=len(LABELS))
# config = LoraConfig(
#     task_type="SEQ_CLS",
#     r=8,
#     lora_alpha=16,
#     lora_dropout=0.05,
#     target_modules=["q_lin", "v_lin"],     # attention projections in DistilBERT
# )
# model = get_peft_model(base, config)
# model.print_trainable_parameters()
# # → 'trainable params: 591k || all params: 67M || trainable %: 0.88'
```

When to reach for LoRA: small training budget, multiple downstream tasks (one LoRA adapter per task), or finetuning anything bigger than ~1 B parameters.


## 7. Inference + saving the fine-tuned model

```python
# Save (LoRA case: only the adapter ~1 MB; full FT: the whole 250 MB model)
# model.save_pretrained("./distilbert-intents")
# tokenizer.save_pretrained("./distilbert-intents")
#
# # Reload at inference time
# clf = AutoModelForSequenceClassification.from_pretrained("./distilbert-intents")
# tok = AutoTokenizer.from_pretrained("./distilbert-intents")
# clf.eval()
# enc = tok("Charge the new card from now on", return_tensors="pt", truncation=True)
# with torch.no_grad():
#     idx = clf(**enc).logits.argmax(dim=1).item()
# print("predicted:", LABELS[idx])
```

**Production tip.** Wrap inference in a `Pipeline` for the simplest API: `pipeline("text-classification", model="./distilbert-intents")`. Returns `[{"label": "billing", "score": 0.97}]`.


## 8. Fine-tune vs prompt-engineer vs RAG — a quick map

Before you fine-tune, ask: do I need to?

| Approach | When it wins |
|---|---|
| **Prompt engineering only** | The base model already knows the domain; you just need format/style nailed (NB 18) |
| **Few-shot in the prompt** | You have a handful of examples and quality has to be high |
| **RAG** (NB 19) | The knowledge is *external* — recent policies, customer-specific facts |
| **Fine-tuning** | The task is *consistent* across all inputs and you have hundreds+ labelled examples |
| **LoRA / PEFT** | Same as fine-tuning but with constrained compute or many task-specific adapters |

In a typical product stack you end up combining **RAG + a lightly fine-tuned classifier** — RAG for the knowledge, fine-tune for routing/labelling.


## 🧪 Exercises

### Exercise 1 — Build the HF fine-tune from scratch in 20 lines
Without copy-pasting from §4, write the smallest end-to-end fine-tune you can: load `distilbert-base-uncased`, build a `Dataset`, train 3 epochs, evaluate on `TEST`. Cap your code at 25 non-blank lines. The discipline reveals which lines are actually load-bearing.


In [6]:
# ── Exercise 1 sketch (run when HF transformers is installed) ───────────
# from transformers import AutoTokenizer, AutoModelForSequenceClassification
# from torch.utils.data import Dataset, DataLoader
#
# tok = AutoTokenizer.from_pretrained("distilbert-base-uncased")
# m   = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=3)
#
# class D(Dataset):
#     def __init__(self, pairs): self.p = pairs
#     def __len__(self): return len(self.p)
#     def __getitem__(self, i):
#         t, l = self.p[i]
#         e = tok(t, return_tensors="pt", padding="max_length", truncation=True, max_length=32)
#         return {k: v.squeeze(0) for k, v in e.items()} | {"labels": torch.tensor(LBL2IDX[l])}
#
# opt = torch.optim.AdamW(m.parameters(), lr=2e-5)
# for ep in range(3):
#     for b in DataLoader(D(TRAIN), batch_size=4, shuffle=True):
#         opt.zero_grad(); m(**b).loss.backward(); opt.step()
# m.eval()
# with torch.no_grad():
#     right = sum((m(**b).logits.argmax(1) == b["labels"]).sum().item()
#                 for b in DataLoader(D(TEST), batch_size=4))
# print("test acc:", right / len(TEST))
print("Sketch above — under 25 lines, full HF fine-tune.")


Sketch above — under 25 lines, full HF fine-tune.


### Exercise 2 — Detect overfitting on tiny data
With only 16 training examples you'll overfit *fast*. Add tracking of train and test loss per epoch, plot them, and identify the epoch where test loss starts climbing while train loss keeps falling. Then add **early stopping** that restores the best weights.


In [7]:
if HAS_TORCH:
    # ── Exercise 2 solution (using the offline stand-in head) ─────────────
    import copy
    model = TinyHead(in_dim=Xtr.shape[1])
    opt = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=1e-4)
    loss_fn = nn.CrossEntropyLoss()
    tr_losses, te_losses = [], []
    best_te, best_state, wait = float("inf"), None, 0
    for epoch in range(200):
        model.train()
        opt.zero_grad(); l_tr = loss_fn(model(Xtr), ytr); l_tr.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            l_te = loss_fn(model(Xte), yte).item()
        tr_losses.append(l_tr.item()); te_losses.append(l_te)
        if l_te < best_te - 1e-4:
            best_te = l_te; best_state = copy.deepcopy(model.state_dict()); wait = 0
        else:
            wait += 1
            if wait >= 20:
                print(f"early stop at epoch {epoch} · best test loss {best_te:.3f}")
                break
    plt.figure(figsize=(7, 3.5))
    plt.plot(tr_losses, label="train"); plt.plot(te_losses, label="test")
    plt.xlabel("epoch"); plt.ylabel("cross-entropy"); plt.legend()
    plt.title("Overfitting watch — diverging curves = bad sign"); plt.show()


## 🧠 Key takeaways

- Fine-tuning = continue training a pretrained model on **your** data with a very small learning rate (~ 2e-5).
- The HuggingFace `transformers` library packages the whole stack: tokenizer + model + scheduler + checkpoints.
- **LoRA / PEFT** is the modern default for anything bigger than a base-size encoder.
- Before fine-tuning, ask: would prompt engineering + RAG already work?

## 🚀 Next step

[`A4_tabpfn_priorlab.ipynb`](./A4_tabpfn_priorlab.ipynb) — TabPFN, the tabular foundation model. Often zero-shot beats gradient boosting on small tabular tasks.
